# ETL — VISp excitatory Patch-seq: Cell Features

Writes 50 `CellFeatureDefinition` rows, one `CellFeatureSet` (`exc_visp_morph_features`), the wide-form morphology feature parquet (389 cells × 50 features), and one `CellFeatureMatrix` pointer. All 389 cells are already registered in `DataItem` by `etl_visp_exc_patchseq_01_dataset_dataitem.ipynb`; no new cell registration is needed. Prerequisite: `etl_visp_exc_patchseq_01_dataset_dataitem.ipynb` (`project_id="visp_patchseq"`, `dataset_id="visp_exc_patchseq"`).

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.arrow_utils import (
    attach_linkml_metadata,
    build_arrow_schema,
    build_cell_feature_matrix_schema,
    models_to_table,
)
from connects_common_connectivity.models import (
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    Unit,
)

In [2]:
DEFS_CSV       = "/data/visp-features-and-mapping/exc_visp_patchseq_morph_feature_definitions.csv"
WIDE_CSV       = "/data/visp-features-and-mapping/morph_features_mMET_exc_wide_unnormalized.csv"
OUTPUT_ROOT    = "../scratch/em_patchseq_wnm_v1/"
PROJECT_ID     = "visp_patchseq"
DATASET_ID     = "visp_exc_patchseq"
FEATURE_SET_ID = "exc_visp_morph_features"

print(f"OUTPUT_ROOT    : {OUTPUT_ROOT}")
print(f"PROJECT_ID     : {PROJECT_ID}")
print(f"DATASET_ID     : {DATASET_ID}")
print(f"FEATURE_SET_ID : {FEATURE_SET_ID}")

OUTPUT_ROOT    : ../scratch/em_patchseq_wnm_v1/
PROJECT_ID     : visp_patchseq
DATASET_ID     : visp_exc_patchseq
FEATURE_SET_ID : exc_visp_morph_features


## Prerequisite check

In [3]:
# Read DataItems registered for the exc dataset via the association table.
assoc = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
    .filter(
        (pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID)
    )
)
assert assoc.shape[0] > 0, (
    f"etl_visp_exc_patchseq_01 must be run first — "
    f"no DataItemDataSetAssociation rows for dataset_id='{DATASET_ID}'"
)
registered_ids = set(assoc["dataitem_id"].to_list())
print(f"Prerequisite OK: {len(registered_ids)} DataItems for dataset_id='{DATASET_ID}'")

# Assert that all wide CSV ids are already registered.
wide_ids_check = pd.read_csv(WIDE_CSV, usecols=["specimen_id"])
all_wide_ids = [str(sid) for sid in wide_ids_check["specimen_id"]]
missing = set(all_wide_ids) - registered_ids
assert not missing, f"{len(missing)} wide CSV ids are not in DataItem: {list(missing)[:5]}"
print(f"All {len(all_wide_ids)} wide CSV cells are registered in DataItem.")

Prerequisite OK: 1528 DataItems for dataset_id='visp_exc_patchseq'
All 389 wide CSV cells are registered in DataItem.


## Load feature definitions

In [4]:
defs_df = pd.read_csv(DEFS_CSV)
print("Definitions shape:", defs_df.shape)
defs_df.head(3)

Definitions shape: (50, 6)


,id,description,unit,data_type,range_min,range_max
0,apical_dendrite_bias_x,Difference in apical dendrite extent in the x-...,MICRONS_LENGTH,<f4,0.0,NaN
1,apical_dendrite_bias_y,Difference in apical dendrite extent in the y-...,MICRONS_LENGTH,<f4,NaN,NaN
2,apical_dendrite_depth_pc_0,First principal component of PCA performed on ...,NONE,<f4,NaN,NaN


## Write `CellFeatureDefinition` rows

In [5]:
feature_defs = []
for _, row in defs_df.iterrows():
    kwargs = dict(
        id=str(row["id"]),
        description=str(row["description"]),
        unit=str(row["unit"]),
        data_type=str(row["data_type"]),
        project_id=PROJECT_ID,
        feature_set_id=FEATURE_SET_ID,
    )
    if pd.notna(row["range_min"]):
        kwargs["range_min"] = float(row["range_min"])
    if pd.notna(row["range_max"]):
        kwargs["range_max"] = float(row["range_max"])
    feature_defs.append(CellFeatureDefinition(**kwargs))

schema_cfd = build_arrow_schema(CellFeatureDefinition)
table_cfd  = attach_linkml_metadata(
    models_to_table(feature_defs, schema=schema_cfd), linkml_class="CellFeatureDefinition"
)
write_deltalake(
    OUTPUT_ROOT + "cellfeaturedefinition/", table_cfd,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND feature_set_id = '{FEATURE_SET_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
print("CellFeatureDefinition written:", table_cfd.shape)

CellFeatureDefinition written: (50, 8)


In [6]:
# Verification
cfd_v = (
    pl.read_delta(OUTPUT_ROOT + "cellfeaturedefinition/")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FEATURE_SET_ID))
)
print(cfd_v.shape); print(cfd_v.head(3))
assert cfd_v.shape[0] == len(feature_defs)
assert cfd_v["id"].n_unique() == len(feature_defs)

(50, 8)
shape: (3, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ id         ┆ descriptio ┆ unit       ┆ data_type ┆ range_min ┆ range_max ┆ project_i ┆ feature_s │
│ ---        ┆ n          ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ d         ┆ et_id     │
│ str        ┆ ---        ┆ str        ┆ str       ┆ f64       ┆ f64       ┆ ---       ┆ ---       │
│            ┆ str        ┆            ┆           ┆           ┆           ┆ str       ┆ str       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ apical_den ┆ Difference ┆ MICRONS_LE ┆ <f4       ┆ 0.0       ┆ null      ┆ visp_patc ┆ exc_visp_ │
│ drite_bias ┆ in apical  ┆ NGTH       ┆           ┆           ┆           ┆ hseq      ┆ morph_fea │
│ _x         ┆ dendrite … ┆            ┆           ┆           ┆           ┆           ┆ tures     │
│ apical_den ┆ Difference ┆ MICRONS_LE ┆ <f4       ┆ null      ┆ null

## Write `CellFeatureSet`

In [7]:
feature_set = CellFeatureSet(
    id=FEATURE_SET_ID,
    description=(
        "Morphological features of excitatory VISp Patch-seq neurons computed from "
        "reconstructed apical and basal dendritic arbors. Used in MET-type analysis "
        "(multimodal electrophysiology, morphology, transcriptomics) to characterize "
        "excitatory cell type diversity in mouse primary visual cortex."
    ),
    feature_definition_ids=[fd.id for fd in feature_defs],
    extraction_method="Computed via https://github.com/AllenInstitute/skeleton_keys.",
    project_id=PROJECT_ID,
)

schema_cfs = build_arrow_schema(CellFeatureSet)
table_cfs  = attach_linkml_metadata(
    models_to_table([feature_set], schema=schema_cfs), linkml_class="CellFeatureSet"
)
write_deltalake(
    OUTPUT_ROOT + "cellfeatureset/", table_cfs,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND id = '{FEATURE_SET_ID}'",
    partition_by=["project_id"],
)
print("CellFeatureSet written:", table_cfs.shape)

CellFeatureSet written: (1, 5)


In [8]:
# Verification
cfs_v = pl.read_delta(OUTPUT_ROOT + "cellfeatureset/").filter(pl.col("id") == FEATURE_SET_ID)
print(cfs_v.shape); print(cfs_v)
assert cfs_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌────────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────┐
│ id                 ┆ description        ┆ feature_definition ┆ extraction_method ┆ project_id    │
│ ---                ┆ ---                ┆ _ids               ┆ ---               ┆ ---           │
│ str                ┆ str                ┆ ---                ┆ str               ┆ str           │
│                    ┆                    ┆ list[str]          ┆                   ┆               │
╞════════════════════╪════════════════════╪════════════════════╪═══════════════════╪═══════════════╡
│ exc_visp_morph_fea ┆ Morphological      ┆ ["apical_dendrite_ ┆ Computed via http ┆ visp_patchseq │
│ tures              ┆ features of exci…  ┆ bias_x", "ap…      ┆ s://github.co…    ┆               │
└────────────────────┴────────────────────┴────────────────────┴───────────────────┴───────────────┘


## Load and write wide-form feature parquet

In [9]:
wide_df = pd.read_csv(WIDE_CSV)
print("Wide CSV shape:", wide_df.shape)

# Rename id column; convert int64 → str to match DataItem ids (values unchanged).
wide_df = wide_df.rename(columns={"specimen_id": "id"})
wide_df["id"] = wide_df["id"].astype(str)
wide_df["project_id"]     = PROJECT_ID
wide_df["feature_set_id"] = FEATURE_SET_ID

# Cast each feature column to its declared data_type.
for fd in feature_defs:
    if fd.id in wide_df.columns:
        wide_df[fd.id] = wide_df[fd.id].astype(np.dtype(fd.data_type))

wide_df.head(3)

Wide CSV shape: (389, 51)


,id,apical_dendrite_bias_x,apical_dendrite_bias_y,apical_dendrite_depth_pc_0,apical_dendrite_depth_pc_1,apical_dendrite_depth_pc_2,apical_dendrite_depth_pc_3,apical_dendrite_early_branch_path,apical_dendrite_emd_with_basal_dendrite,apical_dendrite_extent_x,...,basal_dendrite_soma_percentile_x,basal_dendrite_soma_percentile_y,basal_dendrite_stem_exit_down,basal_dendrite_stem_exit_side,basal_dendrite_stem_exit_up,basal_dendrite_total_length,basal_dendrite_total_surface_area,soma_aligned_dist_from_pia,project_id,feature_set_id
0,601628311,147.076721,388.737335,-0.134846,-7.323705,-16.315250,2.834573,0.377581,63.287174,303.832245,...,0.210607,0.859339,0.00,1.0,0.00,1842.712524,4251.727539,543.539917,visp_patchseq,exc_visp_morph_features
1,603229579,117.918472,513.295654,181.612076,-49.761566,75.880211,18.681379,0.272368,49.580830,323.203125,...,0.468498,0.894992,0.00,1.0,0.00,1559.948242,3349.079102,541.661499,visp_patchseq,exc_visp_morph_features
2,603337985,74.871315,382.300995,-77.738235,-19.625267,-50.744755,-86.666443,0.384813,13.400125,254.446243,...,0.223268,0.305389,0.25,0.5,0.25,1467.914917,2933.822510,458.134216,visp_patchseq,exc_visp_morph_features


In [10]:
schema_wide = build_cell_feature_matrix_schema(feature_set, feature_defs, cell_index_column="id")
table_wide  = pa.Table.from_pandas(wide_df, schema=schema_wide, preserve_index=False)

write_deltalake(
    OUTPUT_ROOT + f"cellfeatures/{FEATURE_SET_ID}/", table_wide,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
print("Wide-form parquet written:", table_wide.shape)

Wide-form parquet written: (389, 53)


In [11]:
# Verification
wide_v = (
    pl.read_delta(OUTPUT_ROOT + f"cellfeatures/{FEATURE_SET_ID}/")
    .filter(pl.col("project_id") == PROJECT_ID)
)
print(wide_v.shape); print(wide_v.head(3))
assert wide_v.shape[0] == len(wide_df)
assert wide_v["id"].n_unique() == len(wide_df), "Duplicate cell ids in wide table"

(389, 53)
shape: (3, 53)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ apical_de ┆ apical_de ┆ apical_de ┆ … ┆ basal_den ┆ soma_alig ┆ project_i ┆ feature_ │
│ ---       ┆ ndrite_bi ┆ ndrite_bi ┆ ndrite_de ┆   ┆ drite_tot ┆ ned_dist_ ┆ d         ┆ set_id   │
│ str       ┆ as_x      ┆ as_y      ┆ pth_pc_0  ┆   ┆ al_surfac ┆ from_pia  ┆ ---       ┆ ---      │
│           ┆ ---       ┆ ---       ┆ ---       ┆   ┆ e_a…      ┆ ---       ┆ str       ┆ str      │
│           ┆ f32       ┆ f32       ┆ f32       ┆   ┆ ---       ┆ f32       ┆           ┆          │
│           ┆           ┆           ┆           ┆   ┆ f32       ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 601628311 ┆ 147.07672 ┆ 388.73733 ┆ -0.134846 ┆ … ┆ 4251.7275 ┆ 543.53991 ┆ visp_patc ┆ exc_visp │
│           ┆ 1         ┆ 5         ┆           ┆   ┆ 39        ┆ 

## Write `CellFeatureMatrix` pointer

In [12]:
output_abs = Path(OUTPUT_ROOT).resolve()
cfm = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FEATURE_SET_ID}",
    feature_set_id=FEATURE_SET_ID,
    parquet_path=f"file://{output_abs}/cellfeatures/{FEATURE_SET_ID}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)

schema_cfm = build_arrow_schema(CellFeatureMatrix)
table_cfm  = attach_linkml_metadata(
    models_to_table([cfm], schema=schema_cfm), linkml_class="CellFeatureMatrix"
)
write_deltalake(
    OUTPUT_ROOT + "cellfeaturematrix/", table_cfm,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND feature_set_id = '{FEATURE_SET_ID}'",
    partition_by=["project_id"],
)
print("CellFeatureMatrix written:", table_cfm.shape)

CellFeatureMatrix written: (1, 5)


In [13]:
# Verification
cfm_v = (
    pl.read_delta(OUTPUT_ROOT + "cellfeaturematrix/")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FEATURE_SET_ID))
)
print(cfm_v.shape); print(cfm_v)
assert cfm_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌────────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────┐
│ id                 ┆ feature_set_id     ┆ parquet_path       ┆ cell_index_column ┆ project_id    │
│ ---                ┆ ---                ┆ ---                ┆ ---               ┆ ---           │
│ str                ┆ str                ┆ str                ┆ str               ┆ str           │
╞════════════════════╪════════════════════╪════════════════════╪═══════════════════╪═══════════════╡
│ visp_patchseq_exc_ ┆ exc_visp_morph_fea ┆ file:///scratch/em ┆ id                ┆ visp_patchseq │
│ visp_morph_f…      ┆ tures              ┆ _patchseq_wn…      ┆                   ┆               │
└────────────────────┴────────────────────┴────────────────────┴───────────────────┴───────────────┘


## Summary

| Output path | Class | Rows |
|---|---|---|
| `cellfeaturedefinition/` | `CellFeatureDefinition` | 50 |
| `cellfeatureset/` | `CellFeatureSet` | 1 (`exc_visp_morph_features`) |
| `cellfeatures/exc_visp_morph_features/` | wide parquet | 389 cells × 50 features |
| `cellfeaturematrix/` | `CellFeatureMatrix` | 1 |

All writes use `mode="overwrite"` with a two-level predicate (`project_id AND feature_set_id`) so re-running is idempotent. The inh Patch-seq notebook (same `project_id`, `feature_set_id='inh_visp_morph_features'`) and any future WNM notebook (same `feature_set_id`, `project_id='visp_wnm'`) cannot clobber these rows.